# 01 - Staging Validation

This notebook validates the cleaned staging parquet files.

The goal is to answer:

- Did cleaning create the expected staging files?
- How many rows are in each staging dataset?
- What date range does each dataset cover?
- Are important fields missing?
- Are IDs duplicated?
- Are statuses clean and understandable?


In [1]:
# pathlib gives readable file path handling.
from pathlib import Path

# pandas is used to read parquet files and display validation tables.
import pandas as pd

# Show more columns so validation outputs are easier to inspect.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

# Detect the project root whether this notebook runs from the root folder or notebooks/.
current_working_directory_path = Path.cwd()

if (current_working_directory_path / "data/staging").exists():
    project_root_path = current_working_directory_path
else:
    project_root_path = current_working_directory_path.parent

# Staging data is the output of the cleaning phase.
staging_data_folder_path = project_root_path / "data/staging"

# These are the expected cleaned files.
orders_staging_file_path = staging_data_folder_path / "orders_clean"
delivery_requests_staging_file_path = staging_data_folder_path / "delivery_requests_clean"
performance_logs_staging_file_path = staging_data_folder_path / "performance_logs_clean"

## 1. Load Staging Files

Run the cleaners before this section:

```powershell
python -m pipeline.transform.clean_orders
python -m pipeline.transform.clean_delivery_requests
python -m pipeline.transform.clean_performance_logs
```

In [2]:
# Read the cleaned orders dataset.
orders_staging = pd.read_parquet(orders_staging_file_path)

# Read the cleaned delivery requests dataset.
delivery_requests_staging = pd.read_parquet(delivery_requests_staging_file_path)

# Read the cleaned performance logs dataset.
performance_logs_staging = pd.read_parquet(performance_logs_staging_file_path)

# Display the basic shape of each staging dataset.
pd.DataFrame([
    {"dataset_name": "orders", "rows": len(orders_staging), "columns": len(orders_staging.columns)},
    {"dataset_name": "delivery_requests", "rows": len(delivery_requests_staging), "columns": len(delivery_requests_staging.columns)},
    {"dataset_name": "performance_logs", "rows": len(performance_logs_staging), "columns": len(performance_logs_staging.columns)},
])

,dataset_name,rows,columns
0,orders,4737516,15
1,delivery_requests,16298890,5
2,performance_logs,3231309,6


## 2. General Validation Helpers

These functions create small validation tables. 

In [3]:
def summarize_dataset(dataframe, dataset_name, id_column_name, timestamp_column_name):
    """Create a compact summary for one staging dataset."""
    
    return {
        "dataset_name": dataset_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "min_timestamp": dataframe[timestamp_column_name].min(),
        "max_timestamp": dataframe[timestamp_column_name].max(),
        "missing_id_count": dataframe[id_column_name].isna().sum(),
        "duplicate_id_count": dataframe[id_column_name].duplicated().sum(),
        "missing_timestamp_count": dataframe[timestamp_column_name].isna().sum(),
    }


def missing_rate_table(dataframe, field_names):
    """Return missing rates for selected fields."""
    
    missing_rate_rows = []
    
    for field_name in field_names:
        missing_rate_rows.append({
            "field_name": field_name,
            "missing_count": dataframe[field_name].isna().sum(),
            "missing_rate": dataframe[field_name].isna().mean(),
        })
    
    return pd.DataFrame(missing_rate_rows).sort_values("missing_rate", ascending=False)


def status_distribution_table(dataframe, status_column_name):
    """Return a distribution table for a status column."""
    
    return (
        dataframe[status_column_name]
        .value_counts(dropna=False)
        .rename_axis(status_column_name)
        .reset_index(name="row_count")
    )

## 3. Dataset Summary

This table checks row counts, date ranges, missing IDs, duplicate IDs, and missing timestamps.

In [4]:
staging_dataset_summary = pd.DataFrame([
    summarize_dataset(orders_staging, "orders", "order_id", "createdAt"),
    summarize_dataset(delivery_requests_staging, "delivery_requests", "request_id", "createdAt"),
    summarize_dataset(performance_logs_staging, "performance_logs", "log_id", "createdAt"),
])

staging_dataset_summary

,dataset_name,rows,columns,min_timestamp,max_timestamp,missing_id_count,duplicate_id_count,missing_timestamp_count
0,orders,4737516,15,2022-01-01 00:01:08.260000+00:00,2026-05-19 12:37:07.822000+00:00,0,0,0
1,delivery_requests,16298890,5,2022-01-01 00:00:06.051000+00:00,2026-05-19 13:49:19.819000+00:00,0,0,0
2,performance_logs,3231309,6,2025-07-17 15:52:01.668000+00:00,2026-05-19 14:02:42.781000+00:00,0,0,0


## 4. Orders Validation

In [5]:
# Inspect the cleaned orders columns.
pd.DataFrame({"orders_column_name": orders_staging.columns})

,orders_column_name
0,order_id
1,branch_id
2,merchant_id
3,createdAt
4,orderStatus
5,area
6,area_original
7,area_quality_flag
8,sub_area
9,city


In [6]:
# Check missing rates for the fields kept in lean staging.
orders_important_field_names = [
    "order_id",
    "createdAt",
    "orderStatus",
    "area",
    "city",
    "area_raw",
    "area_quality_flag",
    "latitude",
    "longitude",
    "branch_id",
    "merchant_id",
    "country",
    "platformName",
]

missing_rate_table(orders_staging, orders_important_field_names)

,field_name,missing_count,missing_rate
12,platformName,2769227,0.584531
3,area,30140,0.006362
4,city,2944,0.000621
5,area_raw,2897,0.000612
9,branch_id,1998,0.000422
1,createdAt,0,0.000000
2,orderStatus,0,0.000000
0,order_id,0,0.000000
6,area_quality_flag,0,0.000000
8,longitude,0,0.000000


In [7]:
# Check order status values after cleaning.
status_distribution_table(orders_staging, "orderStatus")

,orderStatus,row_count
0,completed,4205015
1,canceled,507908
2,failed,24027
3,prepayment,368
4,pending,104
5,en_route,64
6,dispatched,30


In [8]:
# Check area quality flags created by area normalization.
status_distribution_table(orders_staging, "area_quality_flag")

,area_quality_flag,row_count
0,city_as_area,4251573
1,city_as_area_with_sub_area,411547
2,area_raw_as_area,44256
3,invalid_area,30140


## 5. Delivery Requests Validation

In [9]:
# Check missing rates for cleaned delivery request fields.
delivery_request_important_field_names = [
    "request_id",
    "order_id",
    "driver_id",
    "status",
    "createdAt",
]

missing_rate_table(delivery_requests_staging, delivery_request_important_field_names)

,field_name,missing_count,missing_rate
0,request_id,0,0.0
1,order_id,0,0.0
2,driver_id,0,0.0
3,status,0,0.0
4,createdAt,0,0.0


In [10]:
# Check delivery request status values after cleaning.
status_distribution_table(delivery_requests_staging, "status")

,status,row_count
0,pending,9842360
1,accepted,3967462
2,rejected,2489068


## 6. Performance Logs Validation

In [11]:
# Check missing rates for cleaned performance log fields.
performance_log_important_field_names = [
    "log_id",
    "createdAt",
    "endpoint",
    "method",
    "statusCode",
    "responseTime",
]

missing_rate_table(performance_logs_staging, performance_log_important_field_names)

,field_name,missing_count,missing_rate
0,log_id,0,0.0
1,createdAt,0,0.0
2,endpoint,0,0.0
3,method,0,0.0
4,statusCode,0,0.0
5,responseTime,0,0.0


In [12]:
# Check HTTP method distribution after cleaning.
status_distribution_table(performance_logs_staging, "method")

,method,row_count
0,POST,1669771
1,GET,1420218
2,PUT,141299
3,DELETE,10
4,PATCH,7
5,HEAD,4


In [13]:
# Check response time summary after cleaning.
performance_response_time_values = pd.to_numeric(performance_logs_staging["responseTime"], errors="coerce")

pd.DataFrame([{
    "min": performance_response_time_values.min(),
    "p50": performance_response_time_values.quantile(0.50),
    "p95": performance_response_time_values.quantile(0.95),
    "p99": performance_response_time_values.quantile(0.99),
    "max": performance_response_time_values.max(),
}])

,min,p50,p95,p99,max
0,-1891,1767.0,22300.0,39498.0,21485055
